# Evaluate a New Sample

Visualizes the trained model's output on one sample outside the training cohort: clustering, spatial
smoothing, a UMAP, cell-type annotation, and disease prediction. No model forward pass happens here --
everything is either loaded from the embeddings `03_integration/autoencoder.ipynb`'s last section saved,
or computed cheaply from them (clustering, smoothing, a classifier fit on the training cohort's own
saved embeddings). There is no ground truth for a genuinely new sample, so disease prediction here is a
prediction to look at, not an accuracy number.

**Prerequisites**
1. `03_integration/autoencoder.ipynb`, sections 1-7 (or `train_final_model.py`): trains the final model,
   writes `data/combined/03_integration/autoencoders/<bin_size>/weights/dual_ae_latents.npz` -- the
   training cohort's own embeddings, used below to fit the disease classifier.
2. `03_integration/autoencoder.ipynb`, section 8 ("Apply the trained model to a new, unseen sample"):
   encodes the new sample, writes `03_integration/output/<sample>_<unit>_embeddings.npz`.

## Setup

In [ ]:
import sys
from pathlib import Path

repo_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "functions").is_dir())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
sys.path.insert(0, "scripts")

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import scipy.sparse as sp

import evaluate_clustering_quality as ecq
from functions import plotting as pl

SAMPLE = "CRC_PNEW"   # must match the SAMPLE used in autoencoder.ipynb's section 8
UNIT = "nuclei"        # "nuclei" or "bins", must match it too

BLUE = pl.RWTH_CATEGORICAL[0]       # #00549F
RED = pl.RWTH_ACCENT_COLORS[5]      # #CC071E
pd.options.display.float_format = "{:.3f}".format

In [ ]:
npz_path = repo_root / "03_integration/output" / f"{SAMPLE}_{UNIT}_embeddings.npz"
npz = np.load(npz_path)
z_trans, z_token = npz["z_trans"], npz["z_token"]
unit_id, xy = npz["unit_id"], npz["xy"]

print(f"{SAMPLE} ({UNIT}): {len(unit_id):,} units, z_trans {z_trans.shape}, z_token {z_token.shape}")

## Clustering
`sc.pp.neighbors` + `sc.tl.leiden` directly on `z_trans`, same resolution (0.5) used throughout the
thesis evaluation. `flavor="igraph"` instead of the scanpy default `leidenalg`: much faster at this
scale, same convention used everywhere else in this project.

In [ ]:
a_trans = ad.AnnData(z_trans, obs=pd.DataFrame(index=unit_id))

sc.pp.neighbors(a_trans, use_rep="X", n_neighbors=15, random_state=0)
sc.tl.leiden(a_trans, resolution=0.5, random_state=0, flavor="igraph", n_iterations=2)

labels_raw = a_trans.obs["leiden"].to_numpy()
cluster_sizes = pd.Series(labels_raw.astype(int)).value_counts().sort_index()  # leiden labels are strings; sort numerically
cluster_sizes.index.name = "cluster"
cluster_sizes.name = "n_units"
print(f"{len(cluster_sizes)} clusters")
cluster_sizes.to_frame()

## Spatial map: before vs. after embedding smoothing
Each unit's embedding is averaged with its spatial neighbours for 3 rounds, then re-clustered -- the
same operation `evaluate_k_matched_ari.py --smooth-hops` and `03_integration`'s thesis-figure demo use.
A much smaller Leiden resolution is needed afterwards: smoothing removes variance, so the same
resolution would collapse to very few clusters.

In [ ]:
N_HOPS = 3
SMOOTH_RESOLUTION = 0.05  # lower than the raw resolution; adjust if this gives too few/many clusters

conn = ecq.build_slide_graph(UNIT, xy)
A = conn.tocsr() + sp.eye(conn.shape[0])
A = sp.diags(1.0 / np.asarray(A.sum(1)).ravel()) @ A

z_smooth = z_trans.copy()
for _ in range(N_HOPS):
    z_smooth = A @ z_smooth

a_smooth = ad.AnnData(z_smooth, obs=pd.DataFrame(index=unit_id))
sc.pp.neighbors(a_smooth, use_rep="X", n_neighbors=15, random_state=0)
sc.tl.leiden(a_smooth, resolution=SMOOTH_RESOLUTION, random_state=0, flavor="igraph", n_iterations=2)
labels_smooth = a_smooth.obs["leiden"].to_numpy()

print(f"before: {len(np.unique(labels_raw))} clusters, after {N_HOPS} hops: {len(np.unique(labels_smooth))} clusters")

In [ ]:
dot_size = float(np.clip(6000 / len(xy), 2, 40))

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, labels, title in ((axes[0], labels_raw, "before smoothing"), (axes[1], labels_smooth, f"after {N_HOPS}-hop smoothing")):
    a = ad.AnnData(np.zeros((len(labels), 1)), obs=pd.DataFrame({"cluster": pd.Categorical(labels)}))
    a.obsm["spatial"] = xy
    pl.plot_spatial_labels(a, label_key="cluster", ax=ax, point_size=dot_size, title=title, mode="bin")
fig.suptitle(f"{SAMPLE}: cluster reassignment under spatial smoothing")
fig.tight_layout()
plt.show()

## UMAP
Reuses the neighbour graph built for clustering above (`a_trans`, from `sc.pp.neighbors`) rather than
recomputing it; `sc.tl.umap` just needs that graph. Colored by the raw (unsmoothed) clusters.

In [ ]:
sc.tl.umap(a_trans, random_state=0)

fig, ax = plt.subplots(figsize=(7, 6))
pl.plot_spatial_labels(a_trans, spatial_key="X_umap", label_key="leiden", ax=ax,
                       point_size=5, title=f"{SAMPLE}: UMAP of z_trans", invert_y=False, mode="bin")
plt.show()

## Cell-type annotation (CellTypist)
Runs CellTypist's gut-atlas model directly on this sample's real, normalized expression (not a
reconstruction from the embedding), then takes each Leiden cluster's majority predicted type -- same
convention used throughout this project. Requires `pip install celltypist` and the model cached once
(`celltypist.models.download_models(model="Cells_Intestinal_Tract.pkl")`, or copy it from
`~/.celltypist/data/models/` if you already have it from the thesis runs).

In [ ]:
import celltypist

trans_dir = repo_root / "data" / SAMPLE / "01_preprocessing" / ecq.UNITS[UNIT]["dir_cellseg"] / ecq.UNITS[UNIT]["bin_size"] / "custom_binning"
f_trans_new = sc.read_h5ad(trans_dir / "count_area_filtered_adata.h5ad")
f_trans_new.obs.index = f_trans_new.obs["Slide_ID"].astype(str) + "_" + f_trans_new.obs["id"].astype(str)
f_trans_new = f_trans_new[unit_id].copy()  # restrict + reorder to exactly the embedded units

model = celltypist.models.Model.load(model="Cells_Intestinal_Tract.pkl")
predictions = celltypist.annotate(f_trans_new, model=model, majority_voting=False)
unit_cell_type = predictions.predicted_labels["predicted_labels"].to_numpy()

# majority vote per Leiden cluster, not the per-unit call directly -- noisy on sparse counts otherwise
cluster_cell_type = (
    pd.Series(unit_cell_type).groupby(labels_raw).agg(lambda s: s.value_counts().idxmax())
)
cell_type = pd.Series(labels_raw).map(cluster_cell_type).to_numpy()
print(cluster_cell_type.rename("majority cell type").to_frame())

In [ ]:
a_ct = ad.AnnData(np.zeros((len(cell_type), 1)), obs=pd.DataFrame({"cell_type": pd.Categorical(cell_type)}))
a_ct.obsm["spatial"] = xy
fig, ax = plt.subplots(figsize=(8, 8))
pl.plot_spatial_labels(a_ct, label_key="cell_type", ax=ax, point_size=dot_size,
                       title=f"{SAMPLE}: CellTypist cell type (per-cluster majority)", mode="bin")
plt.show()

## Disease prediction
A probe fit on the training cohort's own saved embeddings (`weights/dual_ae_latents.npz`), then applied
to this new sample. Per-unit training labels, not the coarse "every unit of a CRC_ slide is diseased"
label: `CRC_P2`'s own real pathologist annotation where the unit is one of its, label transfer's per-unit
prediction (nearest grid cell's predicted tissue type) everywhere else -- the same convention
`evaluate_disease_signal_ground_truth.py` uses for training, extended to the one slide that actually has
a real annotation. A prediction per unit, not an accuracy check -- there is no ground truth for a
genuinely new sample.

In [ ]:
from sklearn.linear_model import LogisticRegression
from functions import label_transfer as lt
from evaluate_k_matched_ari import ground_truth_reference

weights_dir = ecq.BASEDIR / "combined/03_integration/autoencoders" / ecq.UNITS[UNIT]["bin_size"] / "weights"
cohort = np.load(weights_dir / "dual_ae_latents.npz", allow_pickle=True)
cohort_unit_id, cohort_slide = cohort["unit_id"], cohort["slide"].astype(str)

# per-unit label: CRC_P2's real pathologist annotation for its own units, label transfer's per-unit
# prediction (nearest grid cell) for every other slide -- aligned by unit_id, not by row order, since
# this npz (unlike a LOSO fold's) actually saves it
y_train = np.full(len(cohort_unit_id), -1)
for slide in np.unique(cohort_slide):
    mask = cohort_slide == slide
    cell_xy, trans_sub = ecq.reconstruct_held_out_cells(UNIT, slide)
    pos = pd.Index(trans_sub.obs_names).get_indexer(cohort_unit_id[mask])
    found = pos != -1
    if not found.all():
        print(f"  {slide}: {int((~found).sum())}/{mask.sum()} units not found in current data -- excluded")
    cell_xy_aligned = cell_xy[pos[found]]

    if slide == lt.REFERENCE_SAMPLE:
        tissue = ground_truth_reference(slide, cell_xy_aligned)
    else:
        grid_ref = ecq.load_grid_reference(slide)
        if grid_ref is None:
            print(f"  {slide}: no label-transfer predictions -- excluded")
            continue
        tissue = lt.assign_nearest_grid_label(cell_xy_aligned, grid_ref, label_col=ecq.LABEL_COL)

    idx = np.flatnonzero(mask)[found]
    y_train[idx] = (np.asarray(tissue) == "Neoplasm").astype(int)

labeled = y_train != -1
print(f"training labels: {labeled.sum():,}/{len(y_train):,} units "
      f"({(y_train[labeled] == 1).sum():,} diseased, {(y_train[labeled] == 0).sum():,} normal)")

clf = LogisticRegression(max_iter=5000, class_weight="balanced")
clf.fit(cohort["z_trans"][labeled], y_train[labeled])
p_diseased = clf.predict_proba(z_trans)[:, 1]
disease_label = np.where(p_diseased >= 0.5, "diseased", "normal")

print(f"{(disease_label == 'diseased').mean():.1%} of units predicted diseased "
      f"(mean p={p_diseased.mean():.3f})")

In [ ]:
a_disease = ad.AnnData(np.zeros((len(disease_label), 1)), obs=pd.DataFrame({"status": pd.Categorical(disease_label)}))
a_disease.obsm["spatial"] = xy
fig, ax = plt.subplots(figsize=(8, 8))
pl.plot_spatial_labels(a_disease, label_key="status", ax=ax, point_size=dot_size,
                       palette={"normal": BLUE, "diseased": RED},
                       title=f"{SAMPLE}: predicted disease status", mode="bin")
plt.show()

## Summary

In [ ]:
print(f"{SAMPLE} ({UNIT}), {len(unit_id):,} units")
print(f"  clusters: {len(np.unique(labels_raw))} raw, {len(np.unique(labels_smooth))} after {N_HOPS}-hop smoothing")
print(f"  top cell types: {', '.join(pd.Series(cell_type).value_counts().head(3).index)}")
print(f"  predicted diseased: {(disease_label == 'diseased').mean():.1%}")